In [48]:
# Copyright 2017 The TensorFlow Authors All Rights Reserved.
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.
# =============================================================================


from datetime import datetime
from pathlib import Path

import click
import numpy as np
import pandas as pd
import tensorflow as tf
from pydantic import PositiveInt

import migration.datasets_original_ds as datasets_original_ds
from migration.config import DatasetConfig, OptimizedBound, TrainingConfig
from migration.datasets import TensorflowEncodedBatchedDatasetBuilder
from migration.models import vrnn
from migration.models.vrnn_elbo import VRNN
from migration.models.vrnn_fivo_predict import VRNNboundFIVO

# from tensorflow.keras import mixed_precision


# mixed_precision.set_global_policy('mixed_float16')

def _get_config() -> TrainingConfig:
    return TrainingConfig(
    dataset=DatasetConfig(
        training_parquet='../new_data/ais_train.parquet',
        validation_parquet='../new_data/ais_val.parquet',
        test_parquet='../new_data/ais_test.parquet',
        mean_pickle='../data/ct_2017010203_10_20/mean.pkl',
        shuffle=False,
        val_size=15813 // 32,
        training_size=73795 // 32,
        test_size = 15814 //32
    ), epochs=9999)


def get_pandas_generator(parquet : Path):
    def get_in_memory_dataset_generator():
        _ds = pd.read_parquet(parquet).replace(1, 0.9999)
        _ds.insert(0, 'temp_id', range(0, len(_ds)))
        _ds = _ds.set_index('temp_id', append=True)
        _ds = _ds.sort_index().reset_index(1).drop('temp_id', axis=1)
        _idxs = _ds.index.unique()
        for idx in range(len(_idxs)):
            track = _ds.loc[_idxs[idx]].values
            yield track.reshape((-1, 4)).astype(np.float32)
    return get_in_memory_dataset_generator



def build_test_dataset(cfg : DatasetConfig) -> tf.data.Dataset:
    test_generator = get_pandas_generator(cfg.test_parquet)
    test_dataset = TensorflowEncodedBatchedDatasetBuilder(
        track_generator=test_generator,
        batch_size=cfg.batch_size,
        lat_bins=cfg.encoding_bins.lat,
        lon_bins=cfg.encoding_bins.lon,
        sog_bins=cfg.encoding_bins.sog,
        cog_bins=cfg.encoding_bins.cog,
        shuffle=False,
        repeat=False
    ).build()
    return test_dataset

def create_model(mean_path : Path, latent_size : PositiveInt, total_bins : PositiveInt, bound : OptimizedBound, num_samples : PositiveInt):
    # Convert the mean of the training set to logit space so it can be used to
    # initialize the bias of the generative distribution.
    mean = datasets_original_ds.get_AIS_dataset_mean(mean_path)
    generative_bias_init = -tf.math.log(1. / tf.clip_by_value(mean, 0.0001, 0.9999) - 1)
    generative_distribution_class = vrnn.ConditionalBernoulliDistribution
    if bound == OptimizedBound.elbo:
        model = VRNN(total_bins,
                             latent_size,
                             generative_distribution_class,
                             generative_bias_init=generative_bias_init,
                             raw_sigma_bias=0.5, num_samples=1)
    else: 
        model = VRNNboundFIVO(
            total_bins,
                             latent_size,
                             generative_distribution_class,
                             generative_bias_init=generative_bias_init,
                             raw_sigma_bias=0.5, num_samples=num_samples
        )
    return model

def initialize_model(dataset : tf.data.Dataset, model : tf.keras.Model):
    inputs, targets, lengths = next(iter(dataset))
    model((inputs, targets), lengths)


class InferenceStep:
    def __init__(self, model):
        self.model = model 

    @tf.function(
        input_signature=(
            tf.TensorSpec(shape=[None,32,702], dtype=tf.float32,name='x'),
            tf.TensorSpec(shape=[None,32,702], dtype=tf.float32, name='y'),
            tf.TensorSpec(shape=[32], dtype=tf.int32, name='lengths'),
        )
    )
    def _do(self, x, y, lengths):
        return self.model((x, y),lengths)
    
    def do(self, batch : tuple[tf.Tensor]):
        inputs, targets, lengths = batch
        return self._do(inputs, targets, lengths)



In [ ]:
models_dir = Path("../output/models")
exp = "custom_ds_latent_size_128a_fivo_adamw"

cfg = _get_config()
cfg.model.bound = OptimizedBound.fivo
if cfg.random_seed:
    tf.random.set_seed(cfg.random_seed)

# gather training data
test_dataset = build_test_dataset(cfg.dataset)
# dataset : tf.data.Dataset = create_dataset(cfg.dataset, repeat=False) 


model = create_model(cfg.dataset.mean_pickle, cfg.model.latent_size, cfg.dataset.encoding_bins.total, cfg.model.bound, cfg.model.num_samples)
# needed to initialize the LSTM weights
initialize_model(test_dataset, model)

# loading model
models_dir = models_dir / exp
assert models_dir.exists(), f'experiment with path {models_dir} does not exist' 
model_path = models_dir / "vrnn.weights.h5"
assert model_path.exists(), f'weights file of experiment {models_dir} does not exist' 
model.load_weights(model_path)

: 

In [9]:
steper = InferenceStep(model)

In [10]:
it = iter(test_dataset)

In [11]:
inputs, targets, lengths = next(it)

In [44]:
a = tf.convert_to_tensor(np.ones((15,32))) / tf.convert_to_tensor(2*np.ones((32,)))


In [47]:
a[:,:].assign(a)

AttributeError: 'tensorflow.python.framework.ops.EagerTensor' object has no attribute 'assign'

In [34]:
lengths.set_shape((1,-1))

ValueError: Dimension -1 must be >= 0

In [31]:
tf.reduce_sum(tf.convert_to_tensor([[1,2,3,4],[6,6,7,8]]), axis=1)

<tf.Tensor: shape=(2,), dtype=int32, numpy=array([10, 27], dtype=int32)>

In [27]:
# returns a (max_seq_len, batch_size, num_samples )
a = steper.do((inputs, targets, lengths))

In [28]:
a

<tf.Tensor: shape=(130, 16, 32), dtype=float32, numpy=
array([[[-24.709412  , -35.896683  , -20.961945  , ..., -45.806305  ,
         -60.023315  , -14.082626  ],
        [-20.480957  , -49.90538   , -27.763397  , ..., -47.66765   ,
         -46.809357  , -21.484756  ],
        [-20.729492  , -58.850266  , -22.022293  , ..., -53.343582  ,
         -18.453491  , -21.054886  ],
        ...,
        [-34.939835  , -39.882416  , -22.996796  , ..., -56.82451   ,
         -18.621414  , -18.477982  ],
        [-24.023956  , -21.032272  , -26.497528  , ..., -69.49687   ,
         -43.650864  , -19.96872   ],
        [-25.115952  , -21.183563  , -26.199463  , ..., -51.68869   ,
         -48.164246  , -18.934921  ]],

       [[-10.662857  , -19.547348  , -12.574966  , ...,  -5.812271  ,
         -13.617249  ,  -3.1194    ],
        [-13.179901  ,  -2.988968  , -27.516342  , ...,  -3.4082336 ,
          -2.8373413 ,  -4.546097  ],
        [ -9.09404   ,  -7.957123  , -11.324585  , ...,  -0.830871

In [ ]:
generate 

In [ ]:
import random
import numpy as np
import pyarrow as pa
import pandas as pd

repetitions = 100000000

length = 3
data = {
    'track_id' : np.zeros(repetitions, dtype=np.uint16),
    't' : np.zeros(repetitions, dtype=np.uint16),
    'log_weight': np.zeros(repetitions, dtype=np.float32)

}


for i in range(repetitions):
    data['a'].append(i / 5000)
    data['b'].append(i % 255)
    data['c'].append(i % 2 == 0)


pa_table = pa.Table.from_pydict(data)
pa_table.to_pandas(types_mapper=pd.ArrowDtype)


,a,b,c
0,0.0,0,True
1,0.0002,1,False
2,0.0004,2,True
3,0.0006,3,False
4,0.0008,4,True
...,...,...,...
99999995,19999.999,215,False
99999996,19999.9992,216,True
99999997,19999.9994,217,False
99999998,19999.9996,218,True


In [19]:
import random
import numpy as np
import pyarrow as pa
import pandas as pd

repetitions = 100000000

length = 3
data = {
    'a' : np.zeros(repetitions, dtype=np.float32),
    'b' : np.zeros(repetitions, dtype=np.uint8),
    'c': np.zeros(repetitions, dtype=np.bool_)

}
for i in range(repetitions):
    data['a'][i] = i / 5000
    data['b'][i] = i % 255
    data['c'][i] = i % 2 == 0


pa_table = pa.Table.from_pydict(data)
pa_table.to_pandas(types_mapper=pd.ArrowDtype)


,a,b,c
0,0.0,0,True
1,0.0002,1,False
2,0.0004,2,True
3,0.0006,3,False
4,0.0008,4,True
...,...,...,...
99999995,19999.998047,215,False
99999996,20000.0,216,True
99999997,20000.0,217,False
99999998,20000.0,218,True


In [11]:
df

,a,b,c
0,0.0,0,True
1,0.0002,1,False
2,0.0004,2,True
3,0.0006,3,False
4,0.0008,4,True
...,...,...,...
99995,19.999001,35,False
99996,19.999201,36,True
99997,19.999399,37,False
99998,19.999599,38,True


In [5]:
pa_table

pyarrow.Table
a: float
b: uint8
c: bool
----
a: [[2.3,2.3,2.3,2.3,2.3,...,2.3,2.3,2.3,2.3,2.3]]
b: [[2,2,2,2,2,...,2,2,2,2,2]]
c: [[true,true,true,true,true,...,true,true,true,true,true]]

In [17]:
a.shape

TensorShape([130, 16, 32])